#### Librerias

In [1]:
import pandas as pd

#### Carga de datos raw

In [2]:
df_ccaa = pd.read_csv(
    "../../data/raw_data_viajeros_por_ccaa_07-04-2026.csv",
    sep=";",                    
    encoding="cp1252",
    encoding_errors="replace",
    engine="python",
    on_bad_lines="skip"         
)

df_procedencia = pd.read_csv(
    "../../data/raw_data_procedencia_viajeros_07-04-2026.csv",
    sep=";",                    
    encoding="cp1252",
    encoding_errors="replace",
    engine="python",
    on_bad_lines="skip"         
)

#### Revisión de datos

In [3]:
df_ccaa.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1200 entries, 0 to 1199
Data columns (total 8 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   Viajeros y pernoctaciones         1200 non-null   object 
 1   Residencia: Nivel 1               1200 non-null   object 
 2   Residencia: Nivel 2               1200 non-null   object 
 3   Totales Territoriales             1200 non-null   object 
 4   Comunidades y Ciudades Autónomas  1200 non-null   object 
 5   Provincias                        0 non-null      float64
 6   Periodo                           1200 non-null   object 
 7   Total                             1200 non-null   object 
dtypes: float64(1), object(7)
memory usage: 75.1+ KB


In [4]:
df_ccaa.isnull().sum()

Viajeros y pernoctaciones              0
Residencia: Nivel 1                    0
Residencia: Nivel 2                    0
Totales Territoriales                  0
Comunidades y Ciudades Autónomas       0
Provincias                          1200
Periodo                                0
Total                                  0
dtype: int64

In [5]:
df_procedencia.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3840 entries, 0 to 3839
Data columns (total 5 columns):
 #   Column                     Non-Null Count  Dtype 
---  ------                     --------------  ----- 
 0   Viajeros y pernoctaciones  3840 non-null   object
 1   RESIDENCIA/ORIGEN          3840 non-null   object
 2   Países                     3240 non-null   object
 3   Periodo                    3840 non-null   object
 4   Total                      3622 non-null   object
dtypes: object(5)
memory usage: 150.1+ KB


In [6]:
df_procedencia.isnull().sum()

Viajeros y pernoctaciones      0
RESIDENCIA/ORIGEN              0
Países                       600
Periodo                        0
Total                        218
dtype: int64

In [7]:
df_procedencia['RESIDENCIA/ORIGEN'].value_counts()

RESIDENCIA/ORIGEN
Total                        3360
América (sin EEUU)            120
Estados Unidos de América     120
Japón                         120
Asia (sin Japón)              120
Name: count, dtype: int64

In [8]:
df_procedencia['Países'].value_counts()

Países
Residentes en España           120
Portugal                       120
UE27_2020 sin España           120
UE28 sin España                120
África                         120
Otros Países Europeos          120
Suiza                          120
Rusia                          120
Reino Unido                    120
Noruega                        120
Resto de la U.E.               120
Suecia                         120
República Checa                120
Polonia                        120
Residentes en el Extranjero    120
Países Bajos                   120
Luxemburgo                     120
Italia                         120
Irlanda                        120
Grecia                         120
Francia                        120
Finlandia                      120
Dinamarca                      120
Bélgica                        120
Austria                        120
Alemania                       120
Resto del Mundo                120
Name: count, dtype: int64

#### Transformación

In [9]:
# Convertir nombres de columnas a minúsculas y eliminar espacios
df_procedencia.columns = df_procedencia.columns.str.strip().str.lower()
df_ccaa.columns = df_ccaa.columns.str.strip().str.lower()

In [10]:
# Renombrar columnas para mayor claridad del dato que contienen
df_procedencia.rename(columns={
    "viajeros y pernoctaciones": "tipo"
}, inplace=True)

df_ccaa.rename(columns={
    "viajeros y pernoctaciones": "tipo"
}, inplace=True)

df_procedencia.rename(columns={
    "residencia/origen": "origen",
    "países": "pais"
}, inplace=True)

df_ccaa.rename(columns={
    "residencia: nivel 2": "origen",
    "comunidades y ciudades autónomas": "ccaa"
}, inplace=True)

In [11]:
# Definimos niveles de clasificación de procedencia
df_procedencia["level"] = "unknown"

# Total
df_procedencia.loc[
    df_procedencia["origen"] == "Total",
    "level"
] = "total"

# País
df_procedencia.loc[
    df_procedencia["pais"].notna(),
    "level"
] = "country"

# Región
df_procedencia.loc[
    df_procedencia["level"] == "unknown",
    "level"
] = "region"

In [12]:
# Dropeamos columnas de df_ccaa por solo contener valores nulos o sin utilidad para el análisis
df_ccaa = df_ccaa.drop(columns=["provincias"])
df_ccaa = df_ccaa.drop(columns=["residencia: nivel 1"])

In [13]:
# Extraer año y mes de la columna periodo
df_procedencia["year"] = df_procedencia["periodo"].str[:4].astype(int)
df_procedencia["month"] = df_procedencia["periodo"].str[5:7].astype(int)

df_ccaa["year"] = df_ccaa["periodo"].str[:4].astype(int)
df_ccaa["month"] = df_ccaa["periodo"].str[5:7].astype(int)

df_procedencia

,tipo,origen,pais,periodo,total,level,year,month
0,Viajero,Total,NaN,2015M01,4.251.906,total,2015,1
1,Viajero,Total,NaN,2015M02,4.937.718,total,2015,2
2,Viajero,Total,NaN,2015M03,6.146.078,total,2015,3
3,Viajero,Total,NaN,2015M04,7.668.482,total,2015,4
4,Viajero,Total,NaN,2015M05,9.087.139,total,2015,5
...,...,...,...,...,...,...,...,...
3835,Pernoctaciones,Asia (sin Japón),NaN,2019M08,NaN,region,2019,8
3836,Pernoctaciones,Asia (sin Japón),NaN,2019M09,NaN,region,2019,9
3837,Pernoctaciones,Asia (sin Japón),NaN,2019M10,NaN,region,2019,10
3838,Pernoctaciones,Asia (sin Japón),NaN,2019M11,NaN,region,2019,11


Para poder hacer análisis de series temporales, es útil tener una columna de fecha completa. 

Creamos la columna "date" combinando el año y el mes, y estableciendo el día como 1 (puesto que no tenemos información del día específico).

In [14]:
df_procedencia["date"] = pd.to_datetime(
    df_procedencia["year"].astype(str) + "-" + df_procedencia["month"].astype(str) + "-01"
)

df_ccaa["date"] = pd.to_datetime(
    df_ccaa["year"].astype(str) + "-" + df_ccaa["month"].astype(str) + "-01"
)

In [15]:
# Convertir columnas de total a numéricas
if df_procedencia["total"].dtype == "object":
    df_procedencia["total"] = pd.to_numeric(
        df_procedencia["total"]
            .str.replace(r"\.", "", regex=True)
            .str.replace(",", ".", regex=False),
        errors="coerce"
    )

if df_ccaa["total"].dtype == "object":
    df_ccaa["total"] = pd.to_numeric(
        df_ccaa["total"]
            .str.replace(r"\.", "", regex=True)
            .str.replace(",", ".", regex=False),
        errors="coerce"
    )

##### Países
Eliminados agregados: UE27, UE28
Otros países europeos,
Resto U.E.,
Asia sin Japón, etc.

Eliminado:
“Residentes en el extranjero”

Transformado:
“Residentes en España” → “España”

In [16]:
# 1. Construcción base: pais + fallback a origen
df_procedencia["pais_final"] = df_procedencia["pais"]

mask_null = df_procedencia["pais_final"].isna()
df_procedencia.loc[mask_null, "pais_final"] = df_procedencia.loc[mask_null, "origen"]


# 2. Normalización
df_procedencia["pais_final"] = (
    df_procedencia["pais_final"]
    .str.strip()
    .str.lower()
)


# 3. Mapear España
mask_spain = (
    df_procedencia["pais_final"] == "residentes en españa"
)

df_procedencia.loc[mask_spain, "pais_final"] = "españa"


# 4. Eliminar agregados (ampliado correctamente)
pattern_aggregates = r"(asia|américa|ue\d+|europe|extranjero|total)"

df_procedencia = df_procedencia[
    ~df_procedencia["pais_final"].str.contains(pattern_aggregates, regex=True, na=False)
]

C:\Users\ismae\AppData\Local\Temp\ipykernel_16776\2333486430.py:28: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  ~df_procedencia["pais_final"].str.contains(pattern_aggregates, regex=True, na=False)


In [17]:
for col in ["origen", "pais", "tipo", "level"]:
    print(f"\n--- {col} ---")
    print(df_procedencia[col].value_counts(dropna=False).head(20))


--- origen ---
origen
Total    2760
Japón     120
Name: count, dtype: int64

--- pais ---
pais
Residentes en España    120
Alemania                120
Resto del Mundo         120
África                  120
Suiza                   120
Rusia                   120
Reino Unido             120
Noruega                 120
Resto de la U.E.        120
Suecia                  120
República Checa         120
Portugal                120
Polonia                 120
Países Bajos            120
Luxemburgo              120
Italia                  120
Irlanda                 120
Grecia                  120
Francia                 120
Finlandia               120
Name: count, dtype: int64

--- tipo ---
tipo
Viajero           1440
Pernoctaciones    1440
Name: count, dtype: int64

--- level ---
level
country    2760
region      120
Name: count, dtype: int64


In [18]:
for col in ["ccaa", "origen", "tipo"]:
    print(f"\n--- {col} ---")
    print(df_ccaa[col].value_counts(dropna=False).head(20))


--- ccaa ---
ccaa
01 Andalucía               240
04 Balears, Illes          240
09 Cataluña                240
10 Comunitat Valenciana    240
13 Madrid, Comunidad de    240
Name: count, dtype: int64

--- origen ---
origen
Residentes en España           600
Residentes en el Extranjero    600
Name: count, dtype: int64

--- tipo ---
tipo
Viajero           600
Pernoctaciones    600
Name: count, dtype: int64


In [19]:
print("Procedencia shape:", df_procedencia.shape)
print("CCAA shape:", df_ccaa.shape)

print("\nValores únicos:")
print("Origen procedencia:", df_procedencia["origen"].nunique())
print("Origen CCAA:", df_ccaa["origen"].nunique())
print("CCAA:", df_ccaa["ccaa"].nunique())

Procedencia shape: (2880, 10)
CCAA shape: (1200, 9)

Valores únicos:
Origen procedencia: 2
Origen CCAA: 2
CCAA: 5


In [ ]:
#df_ccaa.to_csv('../../data/clean_data_ccaa_07_04_2026.csv', index=False)
#df_procedencia.to_csv('../../data/clean_data_procedencia_07_04_2026.csv', index=False)

